# Strata Colab Quickstart — Run Qwen3.8 / Qwen3.6 on Google Colab GPUs

Run the **[Strata](https://github.com/Niko1221/Strata)** hybrid GPU/RAM inference engine directly inside a **Google Colab GPU** notebook (`G4`, `H100`, `A100`, `L4`, or `T4`) using fast `HF_XET_HIGH_PERFORMANCE=1` + `STRATA_UV=1` cold starts and the OpenAI-compatible `/v1` API.

- **Auto GPU Model Selection (`MODEL_PRESET = "Auto (match GPU)"`)** — aligned with `lib/common.sh`:
  - **G4 Blackwell (`RTX PRO 6000`, >= 90 GB VRAM)**: `Qwen3.8-Flash-Next` (`NVFP4`, native Blackwell FP4 with `STRATA_PREFILL_NVFP4=w4a4`)
  - **H100 (80 GB) / A100 (40–80 GB, >= 80 GB RAM)**: `Qwen3.8-Flash-Next` (`IQ3_S`)
  - **L4 (23 GB, >= 48 GB RAM)**: `Qwen3.8-Flash-Next` (`IQ2_XS`)
  - **T4 (15 GB) / lower-RAM GPUs**: `Qwen3.6-35B-A3B` (`UD-Q4_K_XL`)
- **Google Drive Sync (`/content/drive/MyDrive/Strata-data`)**: Compatible with **[Strata Colab Desktop](https://github.com/yarik1390/strata-colab-desktop)** (`launcher.bat` / `start.bat` / `./up.sh`). By default (`DRIVE_MODEL_CACHE = False`), Drive syncs per-GPU tuned presets & configs in `<1s` while models download straight from Hugging Face. Set `DRIVE_MODEL_CACHE = True` to also restore/back up only the selected model's files via Drive.
- **Storage & Compute Units**: Verify at least **60–120 GB** of free disk space depending on the selected model (`~23 GB` for `UD-Q4_K_XL`, `~63–84 GB` for `NVFP4` / `IQ2_XS` / `IQ3_S`). Use Colab strictly for **personal interactive research and benchmarking** and delete your runtime when finished.

References: [Strata](https://github.com/Niko1221/Strata) | [Strata Colab Desktop](https://github.com/yarik1390/strata-colab-desktop) | [Google Colab FAQ](https://research.google.com/colaboratory/faq.html)

## 1. Check GPU, RAM & Disk and Clone Strata

Verifies that this Colab runtime has an active NVIDIA GPU, selects a model preset matched to your VRAM/RAM (or your manual dropdown selection, including **`NVFP4 (G4 Blackwell)`**), picks a loopback port (`8089`, `8081`, `8082`, or `8080`), installs `uv`, and clones Strata.

In [ ]:
import json, os, re, shutil, socket, subprocess, sys, urllib.request
from pathlib import Path

MODEL_PRESET = "Auto (match GPU)"  # @param ["Auto (match GPU)", "NVFP4 (G4 Blackwell)", "Qwen3.8-Flash-Next (IQ3_S)", "Qwen3.8-Flash-Next (IQ2_XS)", "Qwen3.8-Flash-Next (Q2_0)", "Qwen3.8-Coder (IQ1_M)", "Qwen3.6-35B-A3B (UD-Q4_K_XL)"]
CONTEXT = "32768"  # @param ["32768", "65536", "131072", "262144"]

GPU_HELP = (
    "No active NVIDIA GPU was detected in this Colab runtime.\n"
    "To switch to a GPU runtime in Google Colab:\n"
    "  1. Click Runtime -> Change runtime type in the top menu.\n"
    "  2. Under Hardware accelerator, select a GPU (G4, H100, A100, L4, or T4).\n"
    "  3. Click Save and re-run this cell."
)

if shutil.which("nvidia-smi") is None:
    print(GPU_HELP)
    raise RuntimeError("GPU preflight failed: nvidia-smi not found. Switch Colab to a GPU runtime.")

smi_proc = subprocess.run(["nvidia-smi"], check=False)
if smi_proc.returncode != 0:
    print(GPU_HELP)
    raise RuntimeError("GPU preflight failed: nvidia-smi returned a non-zero exit code. Switch Colab to a GPU runtime.")

subprocess.run(["free", "-h"], check=True)
subprocess.run(["df", "-h", "/content"], check=True)

try:
    gpu_lines = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"],
        text=True,
    ).strip().splitlines()
except (FileNotFoundError, subprocess.CalledProcessError) as exc:
    print(GPU_HELP)
    raise RuntimeError("GPU preflight failed while querying GPU details via nvidia-smi.") from exc

if not gpu_lines:
    print(GPU_HELP)
    raise RuntimeError("GPU preflight failed: nvidia-smi returned no GPU entries.")

gpu_name, vram_str, driver = [x.strip() for x in gpu_lines[0].split(",")[:3]]
vram_mib = int(float(vram_str))
vram_gb = (vram_mib + 512) // 1024
GPU_SLUG = re.sub(r"[^a-z0-9]+", "-", gpu_name.strip().lower()).strip("-") or "default"

mem_total_kb = 0
with open("/proc/meminfo", encoding="utf-8") as f:
    for line in f:
        if line.startswith("MemTotal:"):
            mem_total_kb = int(line.split()[1])
            break
ram_gib = mem_total_kb / (1024 * 1024)
ram_gb = ((mem_total_kb // 1024) + 512) // 1024

PRESET_MAP = {
    "NVFP4 (G4 Blackwell)": ("nvfp4", "NVFP4"),
    "Qwen3.8-Flash-Next (IQ3_S)": ("", "IQ3_S"),
    "Qwen3.8-Flash-Next (IQ2_XS)": ("", "IQ2_XS"),
    "Qwen3.8-Flash-Next (Q2_0)": ("", "Q2_0"),
    "Qwen3.8-Coder (IQ1_M)": ("coder", "IQ1_M"),
    "Qwen3.6-35B-A3B (UD-Q4_K_XL)": ("qwen36", "UD-Q4_K_XL"),
}

is_blackwell = bool(re.search(r"RTX PRO 6000|Blackwell|B200|B100", gpu_name, re.IGNORECASE))

if MODEL_PRESET == "Auto (match GPU)":
    if is_blackwell and vram_gb >= 90:
        MODEL_FAMILY, MODEL = "nvfp4", "NVFP4"
    elif vram_gb >= 40 and ram_gb >= 80:
        MODEL_FAMILY, MODEL = "", "IQ3_S"
    elif vram_gb >= 22 and ram_gb >= 48:
        MODEL_FAMILY, MODEL = "", "IQ2_XS"
    else:
        MODEL_FAMILY, MODEL = "qwen36", "UD-Q4_K_XL"
else:
    MODEL_FAMILY, MODEL = PRESET_MAP[MODEL_PRESET]

print(f"Detected GPU: {gpu_name} ({vram_mib} MiB / ~{vram_gb} GB VRAM, driver {driver}), System RAM: {ram_gib:.1f} GiB (~{ram_gb} GB)")
print(f"GPU config slug: {GPU_SLUG}")
print(f"Selected preset: {MODEL_PRESET} -> family={MODEL_FAMILY or 'default'}, model={MODEL}, context={CONTEXT}")


def is_strata_port(p: int) -> bool:
    for path in ("/health", "/api/health", "/v1/models"):
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{p}{path}", timeout=1.5) as resp:
                if resp.status == 200:
                    data = json.loads(resp.read().decode("utf-8", errors="ignore"))
                    if not isinstance(data, dict):
                        continue
                    if path == "/v1/models" and isinstance(data.get("data"), list):
                        return True
                    if path != "/v1/models" and (
                        data.get("service") == "strata"
                        or "loaded" in data
                        or "status" in data
                    ):
                        return True
        except Exception:
            pass
    return False


def port_is_free(p: int) -> bool:
    with socket.socket() as sock:
        try:
            sock.bind(("127.0.0.1", p))
            return True
        except OSError:
            return False


PORT = next((p for p in (8089, 8081, 8082, 8080) if is_strata_port(p)), None)
if PORT is None:
    PORT = next((p for p in (8089, 8081, 8082, 8080) if port_is_free(p)), None)
assert PORT is not None, "No free loopback port found among 8089, 8081, 8082, 8080"
os.environ["STRATA_PORT"] = str(PORT)
print("Selected Strata port:", PORT)

if shutil.which("uv") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

root = Path("/content/Strata")
if not (root / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Niko1221/Strata.git", str(root)], check=True)

print("Repository ready. Proceed to the next step.")

## 2. [Optional] Mount Google Drive & Sync Presets / Model Cache

Mounts Google Drive and restores your per-GPU tuned presets & profiles from `/content/drive/MyDrive/Strata-data/strata-config/<gpu>/presets` in `<1s`.

- **`DRIVE_MODEL_CACHE = False` (default)**: Keeps Drive sync instant (presets & configs only) while Step 3 downloads weights at high speed directly from Hugging Face (`HF_XET_HIGH_PERFORMANCE=1`).
- **`DRIVE_MODEL_CACHE = True`**: Also restores only the cached GGUF/pack/drafter files matching your selected `$MODEL` (never bulk-copying unrelated models from `Strata-data/`).

In [ ]:
import os, shutil, time
from pathlib import Path
from google.colab import drive

DRIVE_CACHE_DIR = "/content/drive/MyDrive/Strata-data"  # @param {type:"string"}
DRIVE_MODEL_CACHE = False  # @param {type:"boolean"}

assert "MODEL" in globals() and "GPU_SLUG" in globals(), "Run Cell 1 first to detect your GPU and select a model."

SHARED_KEYWORDS = ("mmproj", "mtp", "ple-fp8", "token-embd", "dense.gguf", "dflash2", ".json")
SKIP_SUFFIXES = (".tmp", ".part", ".incomplete", ".lock")
SKIP_CONFIG_NAMES = {"last-failed-engine.log"}


def matches_model(rel_path: str, model: str) -> bool:
    low = rel_path.lower().replace("\\", "/")
    if low.endswith(SKIP_SUFFIXES) or low.endswith(".done"):
        return False
    if low.startswith("conversation-spill/") or low.startswith("strata-config/"):
        return False
    if model and model.lower() in low:
        return True
    return any(k in low for k in SHARED_KEYWORDS)


def restore_config_dir(src_dir: Path, dst_dir: Path, *, json_only: bool = False) -> int:
    if not src_dir.is_dir():
        return 0
    dst_dir.mkdir(parents=True, exist_ok=True)
    restored = 0
    for src in sorted(src_dir.iterdir()):
        if not src.is_file():
            continue
        fn = src.name
        if fn.startswith(".") or fn in SKIP_CONFIG_NAMES or fn.endswith(SKIP_SUFFIXES):
            continue
        if json_only and not fn.endswith(".json"):
            continue
        dst = dst_dir / fn
        if dst.is_file() and src.stat().st_mtime <= dst.stat().st_mtime + 1e-6:
            continue
        tmp = dst.with_name(dst.name + ".tmp")
        shutil.copy2(src, tmp)
        os.replace(tmp, dst)
        restored += 1
    return restored


def copy_file_with_progress(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    total = src.stat().st_size
    tmp = dst.with_name(dst.name + ".tmp")
    copied = 0
    start_t = time.perf_counter()
    last_print = 0.0
    with src.open("rb") as fsrc, tmp.open("wb") as fdst:
        while True:
            buf = fsrc.read(16 * 1024 * 1024)
            if not buf:
                break
            fdst.write(buf)
            copied += len(buf)
            now = time.perf_counter()
            if total >= 64 * 1024 * 1024 and (now - last_print >= 3.0 or copied == total):
                last_print = now
                rate = copied / 1e6 / max(0.001, now - start_t)
                pct = int(copied * 100 // total)
                print(
                    f"  [Drive -> SSD] {dst.name}: {copied / 1e9:.2f} / {total / 1e9:.2f} GB ({pct}%, {rate:.0f} MB/s)",
                    flush=True,
                )
    os.replace(tmp, dst)
    try:
        shutil.copystat(src, dst)
    except OSError:
        pass


drive.mount("/content/drive")
source = Path(DRIVE_CACHE_DIR)
cfg_root = source / "strata-config" / GPU_SLUG
preset_dst = Path.home() / ".config" / "strata" / "presets"
profile_dst = Path.home() / ".config" / "strata" / "profiles"

restored_cfg = 0
restored_cfg += restore_config_dir(cfg_root / "presets", preset_dst, json_only=False)
restored_cfg += restore_config_dir(cfg_root / "profiles", profile_dst, json_only=True)
if restored_cfg:
    print(f"Restored {restored_cfg} tuned preset/profile file(s) from {cfg_root}")
else:
    print(f"No newer tuned presets found under {cfg_root / 'presets'} (configs will be saved there in Step 8).")

if not DRIVE_MODEL_CACHE:
    print("DRIVE_MODEL_CACHE is False - skipping Drive model copy; Step 3 will download weights directly from Hugging Face.")
elif source.is_dir():
    dest = Path("/content/Strata-data")
    dest.mkdir(parents=True, exist_ok=True)
    candidates = []
    for root_dir, _, files in os.walk(source):
        rel_root = os.path.relpath(root_dir, source).replace("\\", "/")
        if rel_root == "strata-config" or rel_root.startswith("strata-config/") or rel_root.startswith("conversation-spill"):
            continue
        for fn in sorted(files):
            full = Path(root_dir) / fn
            rel = os.path.relpath(full, source).replace("\\", "/")
            if matches_model(rel, MODEL):
                candidates.append(rel)

    if not candidates:
        print(f"No Drive-cached model files matching '{MODEL}' found in {source} - Step 3 will download fresh.")
    else:
        restored_files = 0
        restored_bytes = 0
        for rel in candidates:
            src_f = source / rel
            dst_f = dest / rel
            size = src_f.stat().st_size
            if not (dst_f.is_file() and dst_f.stat().st_size == size):
                print(f"Restoring {rel} ({size / 1e9:.2f} GB)...")
                copy_file_with_progress(src_f, dst_f)
                restored_files += 1
                restored_bytes += size
            src_done = Path(str(src_f) + ".done")
            dst_done = Path(str(dst_f) + ".done")
            if src_done.is_file() and not dst_done.is_file():
                dst_done.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src_done, dst_done)
        print(f"Model-filtered Drive restore complete for '{MODEL}': {restored_files} file(s), {restored_bytes / 1e9:.2f} GB.")
else:
    print("No existing Drive cache directory found at", source, "- Step 3 will download weights fresh.")

## 3. Install Strata and Prepare the Quantized Model

Exports `HF_XET_HIGH_PERFORMANCE=1` and `STRATA_UV=1` for fast cold starts (plus `STRATA_PREFILL_NVFP4=w4a4` when `NVFP4` is selected), applies optional engine flags (`KV_CACHE`, `VISION`, `DRAFTER`), and runs `setup.sh --no-browser --yes --no-start`.

In [ ]:
import os, subprocess
from pathlib import Path

KV_CACHE = "auto"  # @param ["auto", "int8", "q4_0", "k8v4", "fp16"]
VISION = "auto"  # @param ["auto", "yes", "no", "none", "gpu", "cpu"]
DRAFTER = "auto"  # @param ["auto", "mtp", "dflash2"]

assert "MODEL" in globals() and "PORT" in globals(), "Run Cell 1 first to select your model and port."

os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["STRATA_UV"] = "1"
is_nvfp4 = (MODEL.upper() == "NVFP4") or (MODEL_FAMILY.lower() == "nvfp4")
if is_nvfp4:
    os.environ["STRATA_PREFILL_NVFP4"] = os.environ.get("STRATA_PREFILL_NVFP4") or "w4a4"
else:
    os.environ.pop("STRATA_PREFILL_NVFP4", None)

os.environ["PATH"] = (
    "/usr/local/bin:" + os.path.expanduser("~/.local/bin")
    + ":" + os.environ.get("PATH", "")
)
env = os.environ.copy()

cmd = [
    "bash", "setup.sh",
    "--model", MODEL,
    "--context", str(CONTEXT),
    "--port", str(PORT),
    "--data-dir", "/content/Strata-data",
    "--no-browser", "--yes", "--no-start",
]
if MODEL_FAMILY:
    cmd += ["--family", MODEL_FAMILY]
if KV_CACHE and KV_CACHE != "auto":
    cmd += ["--kv", KV_CACHE]
if VISION and VISION != "auto":
    cmd += ["--vision", VISION]
if DRAFTER and DRAFTER != "auto":
    cmd += ["--drafter", DRAFTER]

nvfp4_note = f", STRATA_PREFILL_NVFP4={env['STRATA_PREFILL_NVFP4']}" if is_nvfp4 else ""
print(f"Cold-start env: HF_XET_HIGH_PERFORMANCE=1, STRATA_UV=1{nvfp4_note}")
print("Running:", " ".join(cmd), flush=True)
proc = subprocess.Popen(
    cmd,
    cwd="/content/Strata",
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    errors="replace",
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"setup.sh exited with status {rc}")

candidates = list(Path("/content/Strata/terminal").glob("run-*.sh")) + list(Path("/content/Strata").glob("run-*.sh"))
model_id_hint = f"{MODEL_FAMILY.lower()}-{MODEL.lower()}" if MODEL_FAMILY and MODEL_FAMILY.lower() != "qwen" else MODEL.lower()
exact = [
    p for p in candidates
    if p.name.lower() in (f"run-{MODEL.lower()}.sh", f"run-{model_id_hint}.sh")
]
partial = [p for p in candidates if MODEL.lower() in p.name.lower()]
pool = exact or partial or candidates
pool.sort(key=lambda p: p.stat().st_mtime, reverse=True)

assert pool, "Setup completed but no run-*.sh launch script was found under /content/Strata"
RUN_SCRIPT = pool[0]
print("Setup complete. Launch script:", RUN_SCRIPT)

## 4. Start the Strata Server and Wait for the Model to Load

Launches the generated `run-*.sh` script in the background and polls `http://127.0.0.1:{PORT}/health` (and `/api/health` / `/v1/models`) up to 10 minutes until weights are loaded. If an existing Strata process is already running on `PORT` with a **different** model or context length, this cell stops it cleanly before launching the newly selected model.

In [ ]:
import json, os, re, signal, socket, subprocess, time, requests
from pathlib import Path

assert "PORT" in globals() and "RUN_SCRIPT" in globals(), "Run Cells 1 and 3 first."

BASE = f"http://127.0.0.1:{PORT}"
SPEC_FILE = Path("/content/.strata-running-spec.json")
s = requests.Session()
s.trust_env = False


def get_status():
    health = None
    models = []
    for path in ("/health", "/api/health"):
        try:
            r = s.get(BASE + path, timeout=3)
            if r.ok:
                j = r.json()
                if isinstance(j, dict) and (j.get("service") == "strata" or "loaded" in j or "status" in j):
                    health = j
                    break
        except (requests.RequestException, ValueError):
            pass
    try:
        mr = s.get(BASE + "/v1/models", timeout=3)
        if mr.ok:
            mj = mr.json()
            models = [m["id"] for m in mj.get("data", []) if isinstance(m, dict) and m.get("id")]
    except (requests.RequestException, ValueError):
        pass
    return health, models


def port_in_use(p: int) -> bool:
    with socket.socket() as sock:
        try:
            sock.bind(("127.0.0.1", p))
            return False
        except OSError:
            return True


def find_strata_pids(port: int) -> list[int]:
    pids: set[int] = set()
    my_pid = os.getpid()
    try:
        out = subprocess.check_output(["ss", "-ltnpH", f"sport = :{port}"], text=True, stderr=subprocess.DEVNULL)
        for m in re.finditer(r"pid=(\d+)", out):
            pid = int(m.group(1))
            if pid != my_pid:
                pids.add(pid)
    except Exception:
        pass
    for entry in os.listdir("/proc"):
        if not entry.isdigit():
            continue
        pid = int(entry)
        if pid == my_pid:
            continue
        try:
            cwd = os.readlink(f"/proc/{pid}/cwd")
            exe = os.readlink(f"/proc/{pid}/exe")
            cmd = Path(f"/proc/{pid}/cmdline").read_bytes().decode("utf-8", errors="ignore")
        except OSError:
            continue
        if (
            cwd.startswith("/content/Strata")
            or exe.startswith("/content/Strata")
            or "/content/Strata/" in cmd
            or "strata.py" in cmd
        ):
            pids.add(pid)
    return sorted(pids)


def stop_existing_strata(port: int) -> None:
    prev = globals().get("STRATA_PROC")
    if prev is not None and prev.poll() is None:
        try:
            os.killpg(prev.pid, signal.SIGTERM)
        except OSError:
            pass
    for pid in find_strata_pids(port):
        try:
            os.kill(pid, signal.SIGTERM)
        except OSError:
            pass
    deadline = time.time() + 15.0
    while time.time() < deadline:
        if not find_strata_pids(port) and not port_in_use(port):
            break
        time.sleep(0.5)
    for pid in find_strata_pids(port):
        try:
            os.kill(pid, signal.SIGKILL)
        except OSError:
            pass
    time.sleep(1.0)
    SPEC_FILE.unlink(missing_ok=True)


def running_model_matches(running_model: str, target_model: str, target_family: str) -> bool:
    if SPEC_FILE.is_file():
        try:
            saved = json.loads(SPEC_FILE.read_text(encoding="utf-8"))
            if saved.get("port") == PORT:
                return (
                    str(saved.get("model", "")).lower() == target_model.lower()
                    and str(saved.get("family", "")).lower() == target_family.lower()
                    and str(saved.get("context", "")) == str(CONTEXT)
                    and str(saved.get("run_script", "")) == str(RUN_SCRIPT)
                )
        except Exception:
            pass
    if not running_model:
        return False
    low = running_model.lower()
    if target_model.lower() not in low:
        return False
    fam = target_family.lower()
    if fam == "coder" and "coder" not in low:
        return False
    if fam == "qwen36" and ("qwen3.6" not in low and "qwen36" not in low):
        return False
    if fam == "nvfp4" and "nvfp4" not in low:
        return False
    if fam == "" and any(tok in low for tok in ("coder", "qwen3.6", "qwen36", "nvfp4")):
        return False
    return True


health, models = get_status()
running_model = (health or {}).get("model") or (models[0] if models else "")
server_ready = bool((health and health.get("loaded")) or models)

if server_ready and running_model_matches(running_model, MODEL, MODEL_FAMILY):
    print(f"Reusing already-running Strata server for '{running_model}' on port {PORT}.")
else:
    if server_ready or port_in_use(PORT) or find_strata_pids(PORT):
        print(
            f"Existing Strata process on port {PORT} (model='{running_model or 'unknown'}') "
            f"does not match requested model '{MODEL}' (family='{MODEL_FAMILY or 'default'}'). "
            "Stopping previous server cleanly..."
        )
        stop_existing_strata(PORT)

    assert RUN_SCRIPT.is_file(), f"Launch script missing: {RUN_SCRIPT}"
    launch_env = os.environ.copy()
    launch_env["HF_XET_HIGH_PERFORMANCE"] = "1"
    launch_env["STRATA_UV"] = "1"
    if (MODEL.upper() == "NVFP4") or (MODEL_FAMILY.lower() == "nvfp4"):
        launch_env["STRATA_PREFILL_NVFP4"] = launch_env.get("STRATA_PREFILL_NVFP4") or "w4a4"
    else:
        launch_env.pop("STRATA_PREFILL_NVFP4", None)

    log_path = Path(f"/content/strata-{PORT}.log")
    with log_path.open("a", encoding="utf-8") as log:
        proc = subprocess.Popen(
            ["bash", str(RUN_SCRIPT)],
            cwd=str(RUN_SCRIPT.parent),
            env=launch_env,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    STRATA_PROC = proc
    SPEC_FILE.write_text(
        json.dumps(
            {
                "port": PORT,
                "model": MODEL,
                "family": MODEL_FAMILY,
                "context": str(CONTEXT),
                "run_script": str(RUN_SCRIPT),
                "pid": proc.pid,
            },
            indent=2,
        )
        + "\n",
        encoding="utf-8",
    )
    print(f"Server started (PID={proc.pid}). Loading model weights...")
    for i in range(120):  # up to 10 minutes
        health, models = get_status()
        if (health and health.get("loaded")) or models:
            break
        if proc.poll() is not None:
            print(log_path.read_text(errors="replace")[-5000:])
            raise RuntimeError(f"Strata server exited unexpectedly (exit {proc.returncode})")
        if i % 6 == 0:
            print(f"{i * 5}s elapsed. Waiting for model to finish loading...")
        time.sleep(5)
    else:
        print(log_path.read_text(errors="replace")[-5000:])
        raise TimeoutError("Strata did not become ready within 10 minutes. Check the log above.")

ACTIVE_MODEL_ID = (health or {}).get("model") or (models[0] if models else f"qwen3.8-flash-next-{MODEL.lower()}")
print("READY:", health or {"models": models})
print("OpenAI-compatible API:", BASE + "/v1")
print("Active model ID:", ACTIVE_MODEL_ID)

## 5. Stream Text Generation & Display Speed Metrics

Streams a 512-token response from `/v1/chat/completions` and reports prompt tokens, output tokens, Prefill `tok/s`, Decode `tok/s`, time to first token, and speculative decoding acceptance rate (when active).

In [ ]:
import json, time

payload = {
    "model": ACTIVE_MODEL_ID,
    "messages": [
        {
            "role": "user",
            "content": "Explain with concrete examples how AI is transforming R&D at chemical manufacturers.",
        }
    ],
    "max_tokens": 512,
    "reasoning_effort": "none",
    "temperature": 0,
    "stream": True,
}

start = time.perf_counter()
first_text = None
usage, timings = {}, {}
print("=== Text Generation (Streaming) ===\n")
with s.post(BASE + "/v1/chat/completions", json=payload, stream=True, timeout=(10, 900)) as r:
    if not r.ok:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:800]}")
    for line in r.iter_lines(chunk_size=1):
        if not line or not line.startswith(b"data:"):
            continue
        raw = line[5:].strip()
        if raw == b"[DONE]":
            break
        event = json.loads(raw)
        if event.get("error"):
            raise RuntimeError(str(event["error"]))
        for choice in event.get("choices", []):
            delta = choice.get("delta") or {}
            piece = delta.get("content") or delta.get("reasoning_content") or ""
            if piece:
                if first_text is None:
                    first_text = time.perf_counter()
                print(piece, end="", flush=True)
        usage = event.get("usage") or usage
        timings = event.get("timings") or timings

elapsed = time.perf_counter() - start
print("\n\n=== Benchmark ===")
print("Prompt tokens:", usage.get("prompt_tokens"))
print("Output tokens:", usage.get("completion_tokens"))
print("Prefill:", timings.get("prompt_per_second"), "tok/s")
print("Decode:", timings.get("predicted_per_second"), "tok/s")
print("Total time:", round(elapsed, 2), "s")
if first_text:
    print("Time to first token:", round(first_text - start, 2), "s")
if timings.get("draft_n"):
    accept = timings.get("draft_n_accepted", 0)
    print("Speculative decoding acceptance rate:", round(100 * accept / timings["draft_n"], 1), "%")

## 6. [Optional] Compare Long-Context Prefill Performance

Runs single-shot summarization prompts across **Short (100 chars)**, **Medium (20,000 chars)**, and **Long (50,000 chars)** inputs (`max_tokens=64`) to compare prefill throughput across prompt lengths.

In [ ]:
import time, uuid

paragraph = (
    "Artificial intelligence is transforming scientific research. "
    "Researchers use machine learning to analyze experimental data, "
    "discover new materials, optimize chemical processes, and predict "
    "physical properties. Reliable evaluation requires reproducible "
    "experiments, careful validation, and quantitative comparisons. "
)

for label, chars in [("Short", 100), ("Medium", 20000), ("Long", 50000)]:
    doc = (paragraph * (chars // len(paragraph) + 1))[:chars]
    text = f"Test ID: {uuid.uuid4().hex}\nSummarize the following text concisely.\n{doc}"
    data = {
        "model": ACTIVE_MODEL_ID,
        "messages": [{"role": "user", "content": text}],
        "max_tokens": 64,
        "reasoning_effort": "none",
        "temperature": 0,
        "stream": False,
    }
    t0 = time.perf_counter()
    r = s.post(BASE + "/v1/chat/completions", json=data, timeout=900)
    r.raise_for_status()
    out = r.json()
    tm, usage = out.get("timings", {}), out.get("usage", {})
    print(
        f"{label}: prompt {usage.get('prompt_tokens')} tokens | "
        f"Prefill {tm.get('prompt_per_second')} tok/s | "
        f"Decode {tm.get('predicted_per_second')} tok/s | "
        f"Cache reuse {tm.get('cache_n')} | "
        f"Total time {time.perf_counter() - t0:.2f} s"
    )

## 7. [Optional] Back Up Selected Model Weights to Google Drive

Only needed if you plan to use `DRIVE_MODEL_CACHE = True` in future sessions (or `DRIVE_MODEL_CACHE=1` in `strata-colab-desktop`). Copies only verified model files matching the active `$MODEL` from `/content/Strata-data` to Google Drive. Tuned presets and configs are synced separately in **Step 8**.

In [ ]:
import os, shutil
from pathlib import Path
from google.colab import drive

DRIVE_CACHE_DIR = "/content/drive/MyDrive/Strata-data"  # @param {type:"string"}

assert "MODEL" in globals(), "Run Cell 1 first to set the active MODEL."

drive.mount("/content/drive")
src_root = Path("/content/Strata-data")
dst_root = Path(DRIVE_CACHE_DIR)
dst_root.mkdir(parents=True, exist_ok=True)

SHARED_KEYWORDS = ("mmproj", "mtp", "ple-fp8", "token-embd", "dense.gguf", "dflash2", ".json")
SKIP_SUFFIXES = (".tmp", ".part", ".incomplete", ".lock")

copied_files = 0
copied_bytes = 0
for root_dir, _, files in os.walk(src_root):
    rel_root = os.path.relpath(root_dir, src_root).replace("\\", "/")
    if rel_root.startswith("conversation-spill") or rel_root.startswith("strata-config"):
        continue
    for fn in sorted(files):
        if fn.endswith(SKIP_SUFFIXES):
            continue
        full = Path(root_dir) / fn
        if Path(str(full) + ".part").exists() or Path(str(full) + ".tmp").exists():
            continue
        rel = os.path.relpath(full, src_root).replace("\\", "/")
        low = rel.lower()
        base = low[:-5] if low.endswith(".done") else low
        if not (MODEL.lower() in base or any(k in base for k in SHARED_KEYWORDS)):
            continue
        dst = dst_root / rel
        size = full.stat().st_size
        if dst.is_file() and dst.stat().st_size == size:
            continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        tmp = dst.with_name(dst.name + ".tmp")
        print(f"Backing up {rel} ({size / 1e9:.2f} GB)...", flush=True)
        shutil.copy2(full, tmp)
        os.replace(tmp, dst)
        copied_files += 1
        copied_bytes += size

print(f"Model cache backup complete for '{MODEL}': {copied_files} file(s) ({copied_bytes / 1e9:.2f} GB) -> {dst_root}")

## 8. Stop Server & Sync Presets to Google Drive

Cleanly stops the running Strata server (releasing GPU VRAM and loopback port) and saves any tuned presets & serving profiles from `~/.config/strata/presets` back to `/content/drive/MyDrive/Strata-data/strata-config/<gpu>/presets` so future notebook and desktop runs reuse them automatically.

In [ ]:
import os, re, signal, subprocess, time
from pathlib import Path

DRIVE_CACHE_DIR = "/content/drive/MyDrive/Strata-data"  # @param {type:"string"}
SYNC_PRESETS_TO_DRIVE = True  # @param {type:"boolean"}

# 1. Stop any running Strata server cleanly
my_pid = os.getpid()
stopped_any = False

prev = globals().get("STRATA_PROC")
if prev is not None and prev.poll() is None:
    try:
        os.killpg(prev.pid, signal.SIGTERM)
        stopped_any = True
    except OSError:
        pass


def list_strata_pids() -> list[int]:
    found: set[int] = set()
    port = globals().get("PORT")
    if port is not None:
        try:
            out = subprocess.check_output(["ss", "-ltnpH", f"sport = :{port}"], text=True, stderr=subprocess.DEVNULL)
            for m in re.finditer(r"pid=(\d+)", out):
                pid = int(m.group(1))
                if pid != my_pid:
                    found.add(pid)
        except Exception:
            pass
    for entry in os.listdir("/proc"):
        if not entry.isdigit():
            continue
        pid = int(entry)
        if pid == my_pid:
            continue
        try:
            cwd = os.readlink(f"/proc/{pid}/cwd")
            exe = os.readlink(f"/proc/{pid}/exe")
            cmd = Path(f"/proc/{pid}/cmdline").read_bytes().decode("utf-8", errors="ignore")
        except OSError:
            continue
        if (
            cwd.startswith("/content/Strata")
            or exe.startswith("/content/Strata")
            or "/content/Strata/" in cmd
            or "strata.py" in cmd
        ):
            found.add(pid)
    return sorted(found)


pids = list_strata_pids()
if pids:
    stopped_any = True
    for pid in pids:
        try:
            os.kill(pid, signal.SIGTERM)
        except OSError:
            pass
    deadline = time.time() + 15.0
    while time.time() < deadline and list_strata_pids():
        time.sleep(0.5)
    for pid in list_strata_pids():
        try:
            os.kill(pid, signal.SIGKILL)
        except OSError:
            pass

Path("/content/.strata-running-spec.json").unlink(missing_ok=True)
print("Strata server stopped cleanly." if stopped_any else "No running Strata server found.")

# 2. Sync tuned presets & profiles back to Google Drive
if SYNC_PRESETS_TO_DRIVE:
    gpu_slug = globals().get("GPU_SLUG", "")
    if not gpu_slug:
        try:
            out = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                text=True,
                stderr=subprocess.DEVNULL,
                timeout=5,
            )
            first = out.strip().splitlines()[0].strip().lower() if out.strip() else ""
            gpu_slug = re.sub(r"[^a-z0-9]+", "-", first).strip("-") or "default"
        except Exception:
            gpu_slug = "default"

    drive_mounted = Path("/content/drive/MyDrive").is_dir()
    if not drive_mounted:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            drive_mounted = Path("/content/drive/MyDrive").is_dir()
        except Exception as exc:
            print(f"Skipping Drive preset sync (Drive not mounted: {exc}).")

    if drive_mounted:
        skip_suffixes = (".tmp", ".part", ".incomplete", ".lock")
        skip_names = {"last-failed-engine.log"}
        cfg_root = Path(DRIVE_CACHE_DIR) / "strata-config" / gpu_slug

        def sync_dir(src_dir: Path, dst_dir: Path, *, json_only: bool = False) -> int:
            if not src_dir.is_dir():
                return 0
            dst_dir.mkdir(parents=True, exist_ok=True)
            changed = 0
            for src in sorted(src_dir.iterdir()):
                if not src.is_file():
                    continue
                fn = src.name
                if fn.startswith(".") or fn in skip_names or fn.endswith(skip_suffixes):
                    continue
                if json_only and not fn.endswith(".json"):
                    continue
                payload = src.read_bytes()
                dst = dst_dir / fn
                if dst.is_file() and dst.read_bytes() == payload:
                    continue
                tmp = dst.with_name(dst.name + ".tmp")
                tmp.write_bytes(payload)
                os.replace(tmp, dst)
                changed += 1
            return changed

        synced = 0
        synced += sync_dir(Path.home() / ".config" / "strata" / "presets", cfg_root / "presets", json_only=False)
        synced += sync_dir(Path.home() / ".config" / "strata" / "profiles", cfg_root / "profiles", json_only=True)
        if synced:
            print(f"Synced {synced} tuned preset/profile file(s) to {cfg_root}")
        else:
            print(f"Tuned presets are already up to date (or none created yet) at {cfg_root / 'presets'}")

print("Done. Select Runtime -> Disconnect and delete runtime in Colab to stop Compute Unit billing.")

---

**When finished:** Run **Step 8** to stop Strata and sync tuned presets, then select **Runtime → Disconnect and delete runtime** to stop Compute Unit consumption.

**Credits:** Powered by **[Strata](https://github.com/Niko1221/Strata)** by [@Niko1221](https://github.com/Niko1221) (MIT License), maintained in **[Strata Colab Desktop](https://github.com/yarik1390/strata-colab-desktop)**.